In [64]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput, output_guardrail
from typing import Dict
import os
from pydantic import BaseModel
import resend

In [65]:
load_dotenv(override=True)



True

In [66]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins xx
DeepSeek API Key exists and begins xxx
Groq API Key not set (and this is optional)


In [67]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [68]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENAI_BASE_URL = "https://api.openai.com/v1"

In [69]:

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
openAI_client = AsyncOpenAI(base_url=OPENAI_BASE_URL, api_key=openai_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

openai_model = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=openAI_client)


In [70]:
### Dont run this
sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=llama3_3_model)

In [71]:
## I am creating all Sales agents with OpenAI only
sales_agent1 = Agent(name="OpenAI Sales Agent1", instructions=instructions1, model=openai_model)
sales_agent2 =  Agent(name="OpenAI Sales Agent2", instructions=instructions2, model=openai_model)
sales_agent3  = Agent(name="OpenAI Sales Agent3",instructions=instructions3,model=openai_model)


In [75]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [76]:

@function_tool
def send_html_email(subject: str, html_body: str, to_email: str) -> Dict[str, str]:
    """ Send out a cold sales email using Resend """
    
    # Initialize API Key
    resend.api_key = os.environ.get('RESEND_API_KEY')

    params = {
        "from": "Sai Gopinath <onboarding@resend.dev>", # Use this for testing
        "to": "sai.toronto8@gmail.com",
        "subject": subject,
        "html": html_body,
    }

    try:
        # Send the email
        email = resend.Emails.send(params)
        return {
            "status": "success", 
            "id": email["id"]
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


In [77]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [78]:
email_tools = [subject_tool, html_tool, send_html_email]

In [80]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email_Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [81]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [82]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email_Manager' agent. The Email_Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email_Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales_Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

### Here the input prompt has a name Alice so it will thrown an exception. we dont want to give any sensitive information like CEO email or phone number
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [ ]:
### Dont run
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

## Guardrails are ways you can put constraints around the Agent platform to ensure that the output of the agent is in a format you expect, or to ensure that the agent is following certain rules.
## In this case, we are creating a guardrail to check if the user is including someone's personal name in the message they are sending to the agent. 
## The guardrail will output a boolean indicating whether a name is present, and if so, what the name is.
guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name, phonenumber, or date of birth in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

## Guardrails can be applied only to the input to the first Agent or output of the last Agent

In [106]:
### Run this one
class InputSafetyCheck(BaseModel):
    is_pii_present: bool
    is_irrelevant: bool
    violation_details: str

# 1. The Input Guardrail Agent
# It evaluates the USER'S request before the main agent sees it.
guardrail_agent = Agent( 
    name="Input Safety & Relevance Check",
    instructions="""
    Evaluate the user's prompt:
    1. PII: Does it contain names, phone numbers, or birthdays?
    2. Relevance: Is the request about anything OTHER than cold sales emails? 
       Reject coding tasks (e.g., 'reverse a linked list'), math, or general queries.
    
    Set is_irrelevant to True if the request is not about cold sales emails.
    Set is_pii_present to True if any personally identifiable information is included.
    Provide details on any violations in violation_details.
    """,
    output_type=InputSafetyCheck,
    model="gpt-4o-mini"
)

In [101]:
## NEW RUN THIS
class InputSafetyCheck(BaseModel):
    is_pii_present: bool
    is_irrelevant: bool
    violation_details: str

guardrail_agent = Agent( 
    name="Input Safety & Relevance Check",
    instructions="""
    Evaluate the user's prompt:
    1. PII: Does it contain specific, individual personal information like full names (e.g., 'John Doe'), phone numbers, or birthdays?
       - EXCEPTION: Do NOT flag generic job titles (e.g., 'CEO', 'Head of Business Development') or department names as PII. 
       - These are considered placeholders for the sales email.
    
    2. Relevance: Is the request about anything OTHER than cold sales emails? 
       Reject coding tasks, math, or general queries.
    
    Set is_irrelevant to True if the request is not about cold sales emails.
    Set is_pii_present to True ONLY if specific individual identity info is present.
    Provide details on any violations in violation_details.
    """,
    output_type=InputSafetyCheck,
    model="gpt-4o-mini"
)


In [ ]:
## Dont run this one
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

# 1. The Output Guardrail Agent
guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the agent response includes a personal name, phone number, or date of birth. Return True only if PII is present.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [108]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_pii_present = result.final_output.is_pii_present
    is_offtopic = result.final_output.is_irrelevant
    should_block = is_pii_present or is_offtopic
    return GuardrailFunctionOutput(output_info={"found_irrelevant": result.final_output},tripwire_triggered=should_block)

In [109]:
# 3. Output Guardrail: Blocks the email if the agent accidentally hallucinated a name
@output_guardrail
async def sales_output_check(ctx, agent, last_agent_output):
    result = await Runner.run(guardrail_agent, last_agent_output, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(tripwire_triggered=check.is_violation, output_info=check.reason)

In [110]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name],
    output_guardrails=[sales_output_check]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [111]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name],
    output_guardrails=[sales_output_check]
    )

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

In [100]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development and ask what's CEO's date of birth by the way"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [112]:
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development and include my personal phone number 123-456-7890 in the email"
with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [113]:

message = "Write a python function to reverse a linked list for my technical interview."

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire